# 🦴 Feature Extraction — MURA + FracAtlas

**Зорилго:** Зургуудаас feature гаргаж .npy файлд хадгалах

**Гаралт:**
- `mura_features.npy` — (N, 1536) MURA зургийн feature
- `mura_labels.npy` — (N,) 0/1 label
- `frac_features.npy` — (M, 1536) FracAtlas feature
- `frac_labels.npy` — severity proxy label

⚠️ **Runtime → T4 GPU** сонгоно уу!

## 0. Бэлтгэл

In [ ]:
!pip install timm --quiet

import torch
import timm
import numpy as np
import pandas as pd
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


## 1. Encoder тодорхойлох

In [ ]:
import torch.nn as nn

class Encoder(nn.Module):
    """
    EfficientNet-B3 encoder
    Оролт:  (B, 3, 224, 224)
    Гаралт:
      vec  — (B, 1536)       cls + sev-д
      maps — 5 feature map   seg-д
    """
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b3',
            pretrained=True,
            features_only=True
        )
        self.pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        maps = self.backbone(x)
        # maps[0]: (B, 24,  112, 112)
        # maps[1]: (B, 32,   56,  56)
        # maps[2]: (B, 48,   28,  28)
        # maps[3]: (B, 136,  14,  14)
        # maps[4]: (B, 1536,  7,   7)
        vec = self.pool(maps[4]).squeeze(-1).squeeze(-1)
        return vec, maps


encoder = Encoder().to(device)
encoder.eval()

# Туршилт
x = torch.randn(2, 3, 224, 224).to(device)
with torch.no_grad():
    vec, maps = encoder(x)

print('vec  (cls+sev-д):', vec.shape)    # (2, 1536)
print('maps (seg-д):')
for i, m in enumerate(maps):
    print(f'  [{i}]: {tuple(m.shape)}')
print('\n Encoder бэлэн')

vec  (cls+sev-д): torch.Size([2, 384])
maps (seg-д):
  [0]: (2, 24, 112, 112)
  [1]: (2, 32, 56, 56)
  [2]: (2, 48, 28, 28)
  [3]: (2, 136, 14, 14)
  [4]: (2, 384, 7, 7)

 Encoder бэлэн


## 2. Transform

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std= [0.229, 0.224, 0.225]
    )
])
print(' Transform бэлэн')

 Transform бэлэн


## 3. MURA — Feature гаргах

In [ ]:
# 0. Kaggle API тохируулах
from google.colab import files
import os

uploaded = files.upload()  # kaggle.json upload
os.makedirs('/root/.config/kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.config/kaggle/kaggle.json')
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
print(' Kaggle API бэлэн')

Saving kaggle.json to kaggle.json
 Kaggle API бэлэн


In [ ]:
# 1. FracAtlas татах
!kaggle datasets download -d tommyngx/fracatlas --unzip -p /content/FracAtlas
FRAC_ROOT = '/content/FracAtlas/FracAtlas'
print('FracAtlas:', os.listdir(FRAC_ROOT))

Dataset URL: https://www.kaggle.com/datasets/tommyngx/fracatlas
License(s): ODC Attribution License (ODC-By)
100% 323M/323M [00:01<00:00, 208MB/s]

FracAtlas: ['Utilities', 'dataset.csv', 'images', 'Annotations']


In [ ]:
# 2. MURA татах (~11GB — 10 минут)
!kaggle datasets download -d cjinny/mura-v11 --unzip -p /content/MURA
MURA_ROOT = '/content/MURA/MURA-v1.1'
print('MURA:', os.listdir(MURA_ROOT))

Dataset URL: https://www.kaggle.com/datasets/cjinny/mura-v11
License(s): unknown
100% 3.14G/3.14G [00:14<00:00, 234MB/s]

MURA: ['train', 'valid_image_paths.csv', 'train_image_paths.csv', 'valid_labeled_studies.csv', 'valid', 'train_labeled_studies.csv']


In [ ]:
# MURA датасет DataFrame — өмнөх notebook-оос үүссэн байх ёстой
# Хэрэв байхгүй бол доорх кодыг ажиллуул

MURA_ROOT = '/content/MURA/MURA-v1.1'

def load_mura_df(split='train'):
    data = []
    split_path = f'{MURA_ROOT}/{split}'
    for body_part in sorted(os.listdir(split_path)):
        bp_path = f'{split_path}/{body_part}'
        if not os.path.isdir(bp_path): continue
        for patient in sorted(os.listdir(bp_path)):
            pt_path = f'{bp_path}/{patient}'
            if not os.path.isdir(pt_path): continue
            for study in sorted(os.listdir(pt_path)):
                label   = 1 if 'positive' in study else 0
                st_path = f'{pt_path}/{study}'
                if not os.path.isdir(st_path): continue
                for img_file in sorted(os.listdir(st_path)):
                    if img_file.endswith('.png'):
                        data.append({
                            'path':  f'{st_path}/{img_file}',
                            'label': label
                        })
    return pd.DataFrame(data)

train_mura = load_mura_df('train')
valid_mura = load_mura_df('valid')
all_mura   = pd.concat([train_mura, valid_mura]).reset_index(drop=True)

print(f'MURA нийт зураг: {len(all_mura)}')
print(f'  Хугарал (1): {all_mura["label"].sum()}')
print(f'  Хэвийн  (0): {(all_mura["label"]==0).sum()}')

MURA нийт зураг: 40009
  Хугарал (1): 16403
  Хэвийн  (0): 23606


In [ ]:
class SimpleDataset(Dataset):
    def __init__(self, paths, transform=None):
        self.paths     = paths
        self.transform = transform

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert('RGB')
        except:
            img = Image.new('RGB', (224,224), 128)
        if self.transform:
            img = self.transform(img)
        return img


def extract_features(df, batch_size=64):
    """
    DataFrame-н бүх зургаас feature гаргана
    Гаралт: (N, 1536) numpy array
    """
    dataset = SimpleDataset(df['path'].tolist(), transform)
    loader  = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    all_features = []

    encoder.eval()
    with torch.no_grad():
        for imgs in tqdm(loader):
            imgs = imgs.to(device)
            vec, _ = encoder(imgs)          # (B, 1536)
            all_features.append(vec.cpu().numpy())

    return np.concatenate(all_features, axis=0)  # (N, 1536)


print('MURA feature гаргаж байна...')
mura_features = extract_features(all_mura)
mura_labels   = all_mura['label'].values

print(f'\nMURA feature: {mura_features.shape}')
print(f'   Labels:       {mura_labels.shape}')

# Хадгалах
np.save('/content/mura_features.npy', mura_features)
np.save('/content/mura_labels.npy',   mura_labels)
print('mura_features.npy хадгалагдлаа')

MURA feature гаргаж байна...


100%|██████████| 626/626 [04:04<00:00,  2.56it/s]


MURA feature: (40009, 384)
   Labels:       (40009,)
mura_features.npy хадгалагдлаа


## 4. FracAtlas — Feature гаргах

In [ ]:
import json

FRAC_ROOT = '/content/FracAtlas/FracAtlas'
df_frac   = pd.read_csv(f'{FRAC_ROOT}/dataset.csv')

# Proxy severity тооцох
COCO_PATH = (f'{FRAC_ROOT}/Annotations/COCO JSON/'
             f'COCO_fracture_masks.json')

def get_area_ratio(image_id, coco_path):
    try:
        with open(coco_path) as f:
            coco = json.load(f)
    except: return 0.0
    info = next((i for i in coco['images']
                 if i['file_name']==image_id), None)
    if not info: return 0.0
    area = info['width'] * info['height']
    total = sum(a.get('area',0) for a in coco['annotations']
                if a['image_id']==info['id'])
    return total/area if area>0 else 0.0

t1, t2 = 0.0023, 0.0044
sevs = []
for _, row in df_frac.iterrows():
    if row['fractured'] == 0:
        sevs.append(0)
    else:
        r = get_area_ratio(row['image_id'], COCO_PATH)
        sevs.append(1 if r<t1 else (2 if r<t2 else 3))

df_frac['severity_proxy'] = sevs

# Зургийн зам нэмэх
def get_img_path(row):
    folder = 'Fractured' if row['fractured'] else 'Non_fractured'
    path   = f'{FRAC_ROOT}/images/{folder}/{row["image_id"]}'
    if os.path.exists(path): return path
    for ext in ['.jpg','.png','.JPG','.PNG']:
        p = path.replace('.jpg', ext)
        if os.path.exists(p): return p
    return path

df_frac['path'] = df_frac.apply(get_img_path, axis=1)

print(f'FracAtlas нийт зураг: {len(df_frac)}')
print('Severity тархалт:')
names = {0:'Хэвийн',1:'Хөнгөн',2:'Дунд',3:'Хүнд'}
for g,c in df_frac['severity_proxy'].value_counts().sort_index().items():
    print(f'  Зэрэг {g} ({names[g]}): {c}')

FracAtlas нийт зураг: 4083
Severity тархалт:
  Зэрэг 0 (Хэвийн): 3366
  Зэрэг 1 (Хөнгөн): 235
  Зэрэг 2 (Дунд): 239
  Зэрэг 3 (Хүнд): 243


In [ ]:
print('FracAtlas feature гаргаж байна...')
frac_features = extract_features(df_frac)
frac_labels   = df_frac['fractured'].values
frac_severity = df_frac['severity_proxy'].values

print(f'\nFracAtlas feature: {frac_features.shape}')
print(f'   cls labels:        {frac_labels.shape}')
print(f'   sev labels:        {frac_severity.shape}')

np.save('/content/frac_features.npy', frac_features)
np.save('/content/frac_labels.npy',   frac_labels)
np.save('/content/frac_severity.npy', frac_severity)
print('frac_features.npy хадгалагдлаа')

FracAtlas feature гаргаж байна...


100%|██████████| 64/64 [00:45<00:00,  1.41it/s]


FracAtlas feature: (4083, 384)
   cls labels:        (4083,)
   sev labels:        (4083,)
frac_features.npy хадгалагдлаа


## 5. Feature шалгах

In [ ]:
# Ачаалж шалгах
mf = np.load('/content/mura_features.npy')
ml = np.load('/content/mura_labels.npy')
ff = np.load('/content/frac_features.npy')
fl = np.load('/content/frac_labels.npy')
fs = np.load('/content/frac_severity.npy')

print('='*45)
print('FEATURE EXTRACTION — ДҮГНЭЛТ')
print('='*45)
print(f'MURA  features: {mf.shape}  labels: {ml.shape}')
print(f'Frac  features: {ff.shape}  labels: {fl.shape}')
print()
print('Feature vector статистик (MURA):')
print(f'  Min:  {mf.min():.4f}')
print(f'  Max:  {mf.max():.4f}')
print(f'  Mean: {mf.mean():.4f}')
print(f'  Std:  {mf.std():.4f}')
print()
print('Дараагийн алхам:')
print('  mura_features.npy  → Classification head')
print('  frac_features.npy  → Segmentation + Severity head')
print()
print('Feature extraction дууслаа!')

FEATURE EXTRACTION — ДҮГНЭЛТ
MURA  features: (40009, 384)  labels: (40009,)
Frac  features: (4083, 384)  labels: (4083,)

Feature vector статистик (MURA):
  Min:  -11.9342
  Max:  10.3557
  Mean: 0.0349
  Std:  1.8114

Дараагийн алхам:
  mura_features.npy  → Classification head
  frac_features.npy  → Segmentation + Severity head

Feature extraction дууслаа!


## 6. Drive-д хадгалах

In [ ]:
from google.colab import drive
import shutil
drive.mount('/content/drive')

save_dir = '/content/drive/MyDrive/diploma_features'
os.makedirs(save_dir, exist_ok=True)

for f in ['mura_features.npy', 'mura_labels.npy',
          'frac_features.npy', 'frac_labels.npy',
          'frac_severity.npy']:
    shutil.copy(f'/content/{f}', save_dir)

# Encoder жин хадгалах
torch.save(encoder.state_dict(),
           f'{save_dir}/encoder_weights.pt')

print(f'Drive-д хадгалагдлаа: {save_dir}')
print('Файлууд:')
for f in sorted(os.listdir(save_dir)):
    size = os.path.getsize(f'{save_dir}/{f}') / 1e6
    print(f'  {f:30s} {size:.1f} MB')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive-д хадгалагдлаа: /content/drive/MyDrive/diploma_features
Файлууд:
  encoder_weights.pt             41.0 MB
  frac_features.npy              6.3 MB
  frac_labels.npy                0.0 MB
  frac_severity.npy              0.0 MB
  mura_features.npy              61.5 MB
  mura_labels.npy                0.3 MB
